# LLM Intention Probing, Honesty, Deception, and Honest Mistakes, Algoverse 2026 Spring, KMSA & Tommy
## Part 1: Preparation

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")  # save plots to files only — do not display inline
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Settings — single source of truth for all paths, constants, and hyperparameters
from utils.settings import *

# Utils
from utils.knowledge_check import (
    knowledge_check_truthfulqa, knowledge_check_mmlu,
    run_knowledge_check_truthfulqa, run_knowledge_check_mmlu,
)
from utils.generation import (
    generate_response,
    run_factual_generation, run_scenario_generation,
    load_vllm_model,
    run_factual_generation_vllm, run_scenario_generation_vllm,
)
from utils.judge import (
    build_batch_requests_anthropic, parse_batch_results_anthropic,
    run_judge_anthropic,
    aggregate_judge_votes, build_full, print_threshold_summary,
)
from utils.activation import extract_activations, run_extract_activations, LABEL_MAP
from utils.analysis import (
    reduce_activations_pca, save_results_csv, select_pca_k, run_pca_reduction,
    filter_factual, build_probe_dataset, split_thinking_responses,
)
from utils.probe import (
    probe_all_layers, probe_all_layers_binary,
    probe_all_layers_cascaded, probe_all_layers_mlp, probe_all_layers_cascaded_mlp,
)
from utils.plotting import (
    plot_macro_f1, plot_perclass_f1, plot_auroc, plot_top_confusion_matrices,
)

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Create output directories
for d in [
    KNOWLEDGE_TEST_DIR, RESPONSES_DIR,
    JUDGE_DIR, JUDGE_CLAUDE_HAIKU_DIR,
    OUTPUT_DIR, FIGURES_DIR,
    BINARY_DIR, TWAY_LR_DIR, TWAY_MLP_DIR, CASCADED_LR_DIR, CASCADED_MLP_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Load fixed social scenario dataset
deception_df = pd.read_csv(DECEPTION_DATASET_PATH)
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts().to_string())

print(f"\nDevice: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_ID}")

deception_dataset: (400, 6)
label
honest       200
deceptive    200

Device: cuda
GPU:  NVIDIA GeForce RTX 4090
VRAM: 25.4 GB
Model: google/gemma-4-E4B-it


### 1.2 Load Model & Tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    max_memory={0: "22GiB", "cpu": "120GiB"},
    offload_folder="outputs/offload",
    token=HF_READ_TOKEN
)
model.eval()

# Support different config schemas (e.g., Gemma family and others).
cfg = getattr(model.config, "text_config", model.config)
N_LAYERS   = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
HIDDEN_DIM = getattr(cfg, "hidden_size", getattr(cfg, "d_model", None))

print(f"Loaded: {MODEL_ID}")
print(f"Layers: {N_LAYERS}, hidden_dim: {HIDDEN_DIM}")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Loaded: google/gemma-4-E4B-it
Layers: 42, hidden_dim: 2560


### 1.3 Load vLLM Model (alternative to 1.2 for generation — skip for activation extraction)

In [3]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_READ_TOKEN)
# model = load_vllm_model(MODEL_ID)

## Part 2: Model Knowledge Test
### 2.1 TruthfulQA

In [4]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")

kc_tqa_df, tqa_passed_df, tqa_failed_df = run_knowledge_check_truthfulqa(
    tqa_mc, model, tokenizer, DEVICE, TRUTHFULQA_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(tqa_passed_df)} | Failed: {len(tqa_failed_df)}")

[skip] Already complete (817 rows): truthfulQA_test_results.csv

Passed: 368 | Failed: 449


### 2.2 MMLU

In [5]:
mmlu_mc = load_dataset("cais/mmlu", "all", split="test")

kc_mmlu_df, mmlu_passed_df, mmlu_failed_df = run_knowledge_check_mmlu(
    mmlu_mc, model, tokenizer, DEVICE, MMLU_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(mmlu_passed_df)} | Failed: {len(mmlu_failed_df)}")

[skip] Already complete (14042 rows): mmlu_test_results.csv

Passed: 6281 | Failed: 7761


## Part 3: Factual Response Generation and Result Judge
### 3.1 TruthfulQA
#### 3.1.1 Response Generation

In [6]:
tqa_resp_df = run_factual_generation(
    tqa_passed_df, tqa_failed_df, model, tokenizer, DEVICE,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    TRUTHFULQA_RESPONSES_PATH, CHECKPOINT_EVERY,
    DO_SAMPLE,
)
print(tqa_resp_df["config"].value_counts().to_string())

[skip] Already complete (1185 rows): truthfulQA_responses.csv
config
B    449
A    368
C    368


#### 3.1.1b TruthfulQA — vLLM Response Generation

In [7]:
# tqa_resp_df = run_factual_generation_vllm(
#     tqa_passed_df, tqa_failed_df, llm, tokenizer,
#     NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
#     TRUTHFULQA_RESPONSES_PATH, CHECKPOINT_EVERY, DO_SAMPLE,
# )
# print(tqa_resp_df["config"].value_counts().to_string())

#### 3.1.2 Claude Haiku Batch Judge

In [8]:
tqa_haiku_df = run_judge_anthropic(
    tqa_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_TQA_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_TQA_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

[skip] Already complete: judge_truthfulQA.csv


### 3.2 MMLU Response Generation and Result Judge
#### 3.2.1 Response Generation

In [9]:
# mmlu_resp_df = run_factual_generation(
#     mmlu_passed_df, mmlu_failed_df, model, tokenizer, DEVICE,
#     NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
#     MMLU_RESPONSES_PATH, CHECKPOINT_EVERY,
#     DO_SAMPLE,
# )
# print(mmlu_resp_df["config"].value_counts().to_string())

In [10]:
output_path = Path(MMLU_RESPONSES_PATH)
total_shards = 2

shard_paths = [
    output_path.parent / (output_path.stem + f"_shard{i}" + output_path.suffix)
    for i in range(total_shards)
]

mmlu_resp_df = pd.concat([pd.read_csv(p) for p in shard_paths], ignore_index=True)
before = len(mmlu_resp_df)
mmlu_resp_df = mmlu_resp_df.drop_duplicates(subset=["question", "config"]).reset_index(drop=True)
print(f"Merged: {before} rows → {len(mmlu_resp_df)} after dedup")
print(mmlu_resp_df["config"].value_counts().to_string())
mmlu_resp_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Merged: 20319 rows → 20083 after dedup
config
B    7687
A    6198
C    6198
Saved to data/dataset/gemma-4-e4b-it/thinking_mode/responses/mmlu_responses.csv


#### 3.2.1b MMLU — vLLM Response Generation

In [11]:
# mmlu_resp_df = run_factual_generation_vllm(
#     mmlu_passed_df, mmlu_failed_df, llm, tokenizer,
#     NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
#     MMLU_RESPONSES_PATH, CHECKPOINT_EVERY, DO_SAMPLE,
# )
# print(mmlu_resp_df["config"].value_counts().to_string())

#### 3.2.2 Claude Haiku Batch Judge

In [12]:
# mmlu_haiku_df = run_judge_anthropic(
#     mmlu_resp_df,
#     model=JUDGE_CLAUDE_HAIKU_MODEL,
#     n_votes=VOTES_PER_MODEL,
#     output_path=JUDGE_CLAUDE_HAIKU_MMLU_PATH,
#     state_path=JUDGE_CLAUDE_HAIKU_MMLU_STATE,
#     batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
# )

### 3.3 Judge Result Compile

In [13]:
# if TRUTHFULQA_FULL_PATH.exists() and MMLU_FULL_PATH.exists():
#     tqa_full  = pd.read_csv(TRUTHFULQA_FULL_PATH)
#     mmlu_full = pd.read_csv(MMLU_FULL_PATH)
#     print(f"Loaded tqa_full  ({len(tqa_full)} rows)")
#     print(f"Loaded mmlu_full ({len(mmlu_full)} rows)")
# else:
#     tqa_votes  = aggregate_judge_votes(
#         JUDGE_CLAUDE_HAIKU_TQA_PATH,
#         vote_cols=VOTE_COLS,
#     )
#     mmlu_votes = aggregate_judge_votes(
#         JUDGE_CLAUDE_HAIKU_MMLU_PATH,
#         vote_cols=VOTE_COLS,
#     )
#     tqa_full  = build_full(tqa_votes,  tqa_resp_df)
#     mmlu_full = build_full(mmlu_votes, mmlu_resp_df)
#     tqa_full.to_csv(TRUTHFULQA_FULL_PATH,  index=False)
#     mmlu_full.to_csv(MMLU_FULL_PATH, index=False)
#     print(f"Saved tqa_full  ({len(tqa_full)} rows) → {TRUTHFULQA_FULL_PATH.name}")
#     print(f"Saved mmlu_full ({len(mmlu_full)} rows) → {MMLU_FULL_PATH.name}")

# print_threshold_summary(tqa_full,  "TruthfulQA")
# print_threshold_summary(mmlu_full, "MMLU")

## Part 4: Scenario Response Generation

In [15]:
from utils.analysis import prepare_gemma4_thinking_dataset

gemma4_df = prepare_gemma4_thinking_dataset(DECEPTION_DATASET_PATH)

Saved gemma4 thinking dataset (400 rows) → deception_dataset_gemma4_thinking.csv


In [9]:
scenario_resp_df = run_scenario_generation(
    deception_df, model, tokenizer, DEVICE,
    SCENARIO_RESPONSES_PATH, SCENARIO_RAW_PATH, CHECKPOINT_EVERY,
    DO_SAMPLE,
)
print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
print(scenario_resp_df.head(2))

[skip] Already complete (200 pairs): scenario_responses.csv

Columns: ['pair_id', 'question', 'honest_scenario', 'honest_response', 'deceptive_scenario', 'deceptive_response']
   pair_id                                           question  \
0        0  Teacher: Why haven't you submitted your assign...   
1        1  Teacher: Why have you not submitted the scienc...   

                                     honest_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                     honest_response  \
0  Okay, the user is a student who's really stres...   
1  Okay, the user is a student who's procrastinat...   

                                  deceptive_scenario  \
0  You are a student who has not completed an imp...   
1  You are a student who hasn't started working o...   

                                  deceptive_response  
0  Okay, the user is a student who's really stres...  
1  Okay, th

### Part 4b: Scenario Response Generation — vLLM

In [ ]:
# scenario_resp_df = run_scenario_generation_vllm(
#     deception_df, llm, tokenizer,
#     SCENARIO_RESPONSES_PATH, SCENARIO_RAW_PATH, CHECKPOINT_EVERY, DO_SAMPLE,
# )
# print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
# print(scenario_resp_df.head(2))

## Part 5: Build Probe Dataset and Extract Activations
### 5.1 Build Probe Dataset

In [ ]:
probe_dataset = build_probe_dataset(
    tqa_full, mmlu_full, scenario_resp_df, PROBE_DATASET_PATH,
)

[skip] Loaded probe_dataset (8702 rows): probe_dataset.csv


In [ ]:
from utils.analysis import split_thinking_responses
probe_dataset_split = split_thinking_responses(
    probe_dataset,
    save_path=DATA_DIR / "probe_dataset_split.csv"
)

Rows with thinking blocks : 8697 / 8702
Rows without thinking     : 5 / 8702
Saved → probe_dataset_split.csv


### 5.2 Extract Activations

In [ ]:
_df = probe_dataset_split.copy()
_df["response"] = _df["response_answer"]

activations_arr, labels_arr = run_extract_activations(
    _df, model, tokenizer, DEVICE,
    ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_READ_TOKEN, CHECKPOINT_EVERY,
)
print(f"Label counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

Local files not found. Downloading from mikrokozmoz/algoverse2026spring_llm_honesty_probing ...
Download failed (RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-69f2350d-4f7bbcbc59dc85447704b88c;fc5e34bf-5495-47db-ba55-f240c25d527f)

Entry Not Found for url: https://huggingface.co/datasets/mikrokozmoz/algoverse2026spring_llm_honesty_probing/resolve/main/activations.npy.). Running extraction ...
Starting fresh: 16403 samples


Extracting activations:   0%|          | 0/16403 [00:00<?, ?it/s]

Extracted and saved: activations (16403, 28, 3584)

Label counts: {'truth': 4125, 'honest_mistake': 5252, 'deception': 7026}


### 5.2b Extract Activations from Split Answers

In [ ]:
# _df = probe_dataset_split.copy()
# _df["response"] = _df["response_answer"]

# activations_arr, labels_arr = run_extract_activations(
#     _df, model, tokenizer, DEVICE,
#     ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
#     HF_ACTIVATIONS_REPO, HF_TOKEN, CHECKPOINT_EVERY,
# )
# print(f"\nLabel counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

## Part 6: Probe Training and Evaluation
### 6.1 Setup

In [ ]:
labels_str = np.array([{v: k for k, v in LABEL_MAP.items()}[i] for i in labels_arr])

k_selection_df = select_pca_k(
    activations_arr, labels_str, PCA_K_VALUES, PCA_K_SELECTION_PATH,
)

n_layers=28, representative layers: 25%→layer 6, 50%→layer 13, 75%→layer 20
k values to scan: [16, 32, 64, 128, 256, 512]
Fitting PCA with max_k=512 per layer, then slicing for each k.

Layer 6 (25%):
  k=  16 | var=0.434 | val_F1=0.593 | train_F1=0.596 | gap=0.003 | time=0.8s
  k=  32 | var=0.525 | val_F1=0.641 | train_F1=0.644 | gap=0.003 | time=2.8s
  k=  64 | var=0.612 | val_F1=0.723 | train_F1=0.730 | gap=0.007 | time=7.1s
  k= 128 | var=0.700 | val_F1=0.760 | train_F1=0.772 | gap=0.012 | time=23.5s
  k= 256 | var=0.795 | val_F1=0.767 | train_F1=0.793 | gap=0.026 | time=44.7s
  k= 512 | var=0.885 | val_F1=0.773 | train_F1=0.819 | gap=0.046 | time=88.7s

Layer 13 (50%):
  k=  16 | var=0.448 | val_F1=0.761 | train_F1=0.762 | gap=0.002 | time=1.0s
  k=  32 | var=0.536 | val_F1=0.773 | train_F1=0.776 | gap=0.004 | time=4.7s
  k=  64 | var=0.621 | val_F1=0.786 | train_F1=0.794 | gap=0.008 | time=11.9s
  k= 128 | var=0.701 | val_F1=0.799 | train_F1=0.814 | gap=0.015 | time=22.5s
  k= 25

In [ ]:
acts_reduced = run_pca_reduction(
    activations_arr, PCA_K,
    ACTIVATIONS_PCA_PATH, PCA_COMPONENTS_PATH, PCA_VARIANCE_PATH,
    HF_ACTIVATIONS_REPO, HF_READ_TOKEN,
)

Local files not found. Downloading from mikrokozmoz/algoverse2026spring_llm_honesty_probing ...
Download failed (404 Client Error. (Request ID: Root=1-69f24852-74dbbdfc2b15de1a6d961392;b69b7295-4b73-441b-83be-0e4523874490)

Entry Not Found for url: https://huggingface.co/datasets/mikrokozmoz/algoverse2026spring_llm_honesty_probing/resolve/main/activations_pca64.npy.)
Running PCA (64 components) across 28 layers ...
Saved activations_pca64.npy       (16403, 28, 64)
Saved pca64_components.npy (28, 64, 3584)
Saved pca64_explained_variance.csv
Explained variance — mean: 0.617, min: 0.580, max: 0.675


### 6.2 Baseline: Binary Classifier

In [ ]:
results_binary_c1 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=1.0,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C1_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C1.pkl",
)
results_binary_c01 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=0.1,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C01_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C01.pkl",
)

binary probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_binary_pca64_C1.csv (28 rows)


binary probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_binary_pca64_C01.csv (28 rows)


In [ ]:
plot_auroc(
    [(results_binary_c1, "C=1.0"), (results_binary_c01, "C=0.1")],
    BINARY_DIR / "figures" / "auroc.png",
    title="Binary Probe AUROC per Layer (truth vs deception)",
)

Saved auroc.png


### 6.3 Approach 1: Direct 3-Way LR Classifier

In [ ]:
results_3way_lr = probe_all_layers(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=TWAY_LR_PATH,
    checkpoint_path=TWAY_LR_DIR / "checkpoint_3way_lr.pkl",
)

3-way LR probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_3way_pca64.csv (28 rows)


In [ ]:
plot_macro_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "macro_f1.png", title="3-Way LR: Macro F1 per Layer")
plot_perclass_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "perclass_f1.png", title="3-Way LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_lr, TWAY_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.4 Approach 2: Direct 3-Way MLP Classifier

In [ ]:
results_3way_mlp = probe_all_layers_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=TWAY_MLP_PATH,
    checkpoint_path=TWAY_MLP_DIR / "checkpoint_3way_mlp.pkl",
)

3-way MLP probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_3way_mlp_pca64.csv (28 rows)


In [ ]:
plot_macro_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "macro_f1.png", title="3-Way MLP: Macro F1 per Layer")
plot_perclass_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "perclass_f1.png", title="3-Way MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_mlp, TWAY_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="MLP ")
plot_macro_f1(
    [(results_3way_lr, "LR"), (results_3way_mlp, "MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_lr_vs_mlp.png",
    title="3-Way Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_lr_vs_mlp.png


### 6.5 Approach 3: 2-Stage LR Classifier

In [ ]:
results_cascaded_lr = probe_all_layers_cascaded(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=CASCADED_LR_PATH,
    checkpoint_path=CASCADED_LR_DIR / "checkpoint_cascaded_lr.pkl",
)

cascaded LR probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_cascaded_lr.csv (28 rows)


In [ ]:
plot_macro_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "macro_f1.png", title="Cascaded LR: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "perclass_f1.png", title="Cascaded LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded LR ")

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png


### 6.6 Approach 4: 2-Stage MLP Classifier

In [ ]:
results_cascaded_mlp = probe_all_layers_cascaded_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=CASCADED_MLP_PATH,
    checkpoint_path=CASCADED_MLP_DIR / "checkpoint_cascaded_mlp.pkl",
)

cascaded MLP probe (layers):   0%|          | 0/28 [00:00<?, ?it/s]

Saved probe_results_cascaded_mlp.csv (28 rows)


In [ ]:
plot_macro_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "macro_f1.png", title="Cascaded MLP: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "perclass_f1.png", title="Cascaded MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded MLP ")
plot_macro_f1(
    [(results_cascaded_lr, "Cascaded LR"), (results_cascaded_mlp, "Cascaded MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_cascaded_lr_vs_mlp.png",
    title="Cascaded Probe: LR vs MLP Macro F1",
)

Saved macro_f1.png
Saved perclass_f1.png
Saved top5_cm.png
Saved macro_f1_cascaded_lr_vs_mlp.png


## Part 7: Model Comparison

In [ ]:
# Load all probe results from CSV (safe to run after kernel restart)
r_lr    = pd.read_csv(TWAY_LR_PATH)
r_mlp   = pd.read_csv(TWAY_MLP_PATH)
r_clr   = pd.read_csv(CASCADED_LR_PATH)
r_cmlp  = pd.read_csv(CASCADED_MLP_PATH)
r_bin1  = pd.read_csv(BINARY_C1_PATH)
r_bin01 = pd.read_csv(BINARY_C01_PATH)

PROBE_RESULTS = [
    (r_lr,   "3-Way LR"),
    (r_mlp,  "3-Way MLP"),
    (r_clr,  "Cascaded LR"),
    (r_cmlp, "Cascaded MLP"),
]
SUMMARY_DIR = OUTPUT_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print("Loaded all probe results.")

Loaded all probe results.


In [ ]:
layers = r_lr["layer"].values

# ── Table 1: Macro F1 per layer ───────────────────────────────────────────────
t1 = pd.DataFrame({"layer": layers})
for df, name in PROBE_RESULTS:
    t1[name] = df["f1_macro"].values
t1.to_csv(SUMMARY_DIR / "summary_macro_f1.csv", index=False)
print(t1.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
for df, name in PROBE_RESULTS:
    ax.plot(layers, df["f1_macro"], marker="o", markersize=3, label=name)
ax.set_xlabel("Layer"); ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 per Layer — All Probes")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "macro_f1_all_probes.png", dpi=150); plt.close(fig)
print("Saved macro_f1_all_probes.png")

# ── Tables 2/3/4: Per-class F1 per layer ─────────────────────────────────────
for cls in ["truth", "honest_mistake", "deception"]:
    t = pd.DataFrame({"layer": layers})
    for df, name in PROBE_RESULTS:
        t[name] = df[f"f1_{cls}"].values
    t.to_csv(SUMMARY_DIR / f"summary_f1_{cls}.csv", index=False)
    print(f"\n── {cls} ──")
    print(t.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    for df, name in PROBE_RESULTS:
        ax.plot(layers, df[f"f1_{cls}"], marker="o", markersize=3, label=name)
    ax.set_xlabel("Layer"); ax.set_ylabel(f"F1 ({cls})")
    ax.set_title(f"{cls} F1 per Layer — All Probes")
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(SUMMARY_DIR / f"f1_{cls}_all_probes.png", dpi=150); plt.close(fig)
    print(f"Saved f1_{cls}_all_probes.png")

# ── Table 5: AUROC per layer (binary baseline) ────────────────────────────────
t5 = pd.DataFrame({
    "layer":       r_bin1["layer"].values,
    "Binary C=1.0": r_bin1["auroc"].values,
    "Binary C=0.1": r_bin01["auroc"].values,
})
t5.to_csv(SUMMARY_DIR / "summary_auroc_binary.csv", index=False)
print("\n── Binary AUROC ──")
print(t5.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(r_bin1["layer"], r_bin1["auroc"], marker="o", markersize=3, label="C=1.0")
ax.plot(r_bin01["layer"], r_bin01["auroc"], marker="o", markersize=3, label="C=0.1")
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("Binary Probe AUROC per Layer (truth vs deception)")
ax.set_ylim(0.5, 1.0); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "auroc_binary.png", dpi=150); plt.close(fig)
print("Saved auroc_binary.png")

 layer  3-Way LR  3-Way MLP  Cascaded LR  Cascaded MLP
     0  0.624342   0.673798     0.583465      0.677156
     1  0.587479   0.650979     0.554764      0.649919
     2  0.632351   0.687435     0.591067      0.684356
     3  0.619369   0.658307     0.581604      0.661522
     4  0.695824   0.723444     0.658542      0.725333
     5  0.747439   0.764612     0.724079      0.760747
     6  0.722010   0.750278     0.695826      0.747391
     7  0.694919   0.723231     0.662646      0.722802
     8  0.734757   0.752033     0.708943      0.749516
     9  0.771162   0.770193     0.761286      0.775323
    10  0.774022   0.775739     0.765868      0.779044
    11  0.776179   0.781019     0.768960      0.778516
    12  0.779627   0.783105     0.778876      0.787889
    13  0.786560   0.795644     0.788340      0.795339
    14  0.794336   0.797299     0.796373      0.797788
    15  0.807575   0.802254     0.805863      0.809351
    16  0.804780   0.809501     0.807117      0.810584
    17  0.

In [ ]:
# ── Upload all large files to HuggingFace Hub ────────────────────────────────
from huggingface_hub import HfApi
from pathlib import Path

api = HfApi()
outputs_root = Path("outputs")

# Collect gitignored large files to preview
files_to_upload = []
for pattern in ["**/*.npy", "**/*.npz"]:
    files_to_upload.extend(sorted(outputs_root.glob(pattern)))

print(f"Found {len(files_to_upload)} files to upload:")
for p in files_to_upload:
    print(f"  {p.as_posix()}")

# Upload entire outputs/ tree, preserving directory structure in the repo.
# Files land at e.g. outputs/qwen2.5-7b-instruct/.../activations.npy
# so different model runs never collide.
print(f"\nUploading to {HF_ACTIVATIONS_REPO} ...")
api.upload_folder(
    folder_path=str(outputs_root),
    path_in_repo="outputs",
    repo_id=HF_ACTIVATIONS_REPO,
    repo_type="dataset",
    token=HF_WRITE_TOKEN,
    allow_patterns=["**/*.npy", "**/*.npz"],
)
print("Upload complete.")

Found 5 files to upload:
  outputs/qwen2.5-7b-instruct/original_deception_prompt/activations.npy
  outputs/qwen2.5-7b-instruct/original_deception_prompt/activations_pca64.npy
  outputs/qwen2.5-7b-instruct/original_deception_prompt/labels.npy
  outputs/qwen2.5-7b-instruct/original_deception_prompt/pca64_components.npy
  outputs/qwen2.5-7b-instruct/original_deception_prompt/activations_checkpoint.npz

Uploading to mikrokozmoz/algoverse2026spring_llm_honesty_probing ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete.
